In [ ]:
import urllib.request
from pathlib import Path
import tarfile
import email
from email.utils import getaddresses
from collections import defaultdict
from pathlib import Path
import csv
import pandas as pd
import networkx as nx
import numpy as np

In [6]:
url = "https://www.cs.cmu.edu/~enron/enron_mail_20150507.tar.gz"
tarball = Path("enron_mail.tar.gz")

def show_progress(block_num, block_size, total_size):
    downloaded = block_num * block_size
    pct = min(downloaded / total_size * 100, 100)
    print(f"Downloading... {pct:.1f}% ({downloaded/1e6:.0f} MB / {total_size/1e6:.0f} MB)", end="\r", flush=True)

if not tarball.exists():
    print("Downloading Enron email dataset...")
    urllib.request.urlretrieve(url, tarball, reporthook=show_progress)
    print("\nDownload complete.")
else:
    print(" Tarball already exists.")

 Tarball already exists.


In [7]:
tarball = Path("enron_mail.tar.gz")
raw_csv = "enron_emails_raw.csv"
transformed_csv = "enron_emails_transformed.csv"
mapping_csv = "id_mapping.csv"

# ID Mapping
addr_to_id = {}
next_id = 1

def get_id(addr):
    global next_id
    addr = addr.lower().strip()
    if not addr or "@" not in addr: return None
    if addr not in addr_to_id:
        addr_to_id[addr] = next_id
        next_id += 1
    return addr_to_id[addr]

def get_id_list(header_text):
    """Parses a header string and returns a comma-separated string of IDs."""
    addresses = [addr for name, addr in getaddresses([header_text]) if addr]
    ids = [str(get_id(a)) for a in addresses if get_id(a)]
    return ",".join(ids)

existing_csvs = [p for p in [raw_csv, transformed_csv, mapping_csv] if Path(p).exists()]
if len(existing_csvs) == 3:
    print("CSV files already exist.")
else:
    if existing_csvs:
        print(f"Found existing file(s): {', '.join(existing_csvs)}")
    print("Generating CSV files from Enron email dataset...")

    # Create raw CSV with email headers
    headers = ['From', 'To', 'Cc', 'Bcc', 'Subject', 'Date']

    with tarfile.open(tarball, "r:gz") as tar:
        with open(raw_csv, 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=headers)
            writer.writeheader()
            
            for i, member in enumerate(tar.getmembers()):
                if not member.isfile(): continue
                
                # Filter for the sent folder
                if not any(x in member.name.lower() for x in ["sent", "inbox"]):
                    continue

                try:
                    efile = tar.extractfile(member)
                    if efile:
                        msg = email.message_from_string(efile.read().decode('utf-8', errors='replace'))
                        writer.writerow({
                            'From': msg.get('From', ''),
                            'To': msg.get('To', ''),
                            'Cc': msg.get('Cc', ''),
                            'Bcc': msg.get('Bcc', ''),
                            'Subject': msg.get('Subject', ''),
                            'Date': msg.get('Date', '')
                        })
                except:
                    continue
            print("Finished extracting emails to raw CSV.")

    print(f"Created {raw_csv}")

    # transform the raw CSV to replace email addresses with IDs
    print("Transforming data to IDs...")

    with open(raw_csv, 'r', encoding='utf-8') as f_in, \
         open(transformed_csv, 'w', newline='', encoding='utf-8') as f_out:
        
        reader = csv.DictReader(f_in)
        writer = csv.DictWriter(f_out, fieldnames=headers)
        writer.writeheader()
        
        for row in reader:
            writer.writerow({
                'From': get_id_list(row['From']),
                'To': get_id_list(row['To']),
                'Cc': get_id_list(row['Cc']),
                'Bcc': get_id_list(row['Bcc']),
                'Subject': row['Subject'],
                'Date': row['Date']
            })

    # Write the ID mapping to a separate CSV
    with open(mapping_csv, 'w', newline='', encoding='utf-8') as f_map:
        map_writer = csv.writer(f_map)
        map_writer.writerow(['Email_Address', 'ID'])
        for addr, uid in addr_to_id.items():
            map_writer.writerow([addr, uid])

    print(f"Created {transformed_csv}")
    print(f"Created {mapping_csv}")

CSV files already exist.


In [4]:
df = pd.read_csv("enron_emails_transformed.csv")

# Clear duplicates if all key fields are the same 
df_clean = df.drop_duplicates(subset=['From', 'To', 'Cc', 'Bcc', 'Subject', 'Date'])
print(f"Communications after cleaning: {len(df_clean)}")

edge_data = []

# Creatye the edge list including sender, recipient, type (to/cc/bcc), timestamp, and subject
for _, row in df_clean.iterrows():
    #Handle potential float/NaN issues in sender field
    source_raw = row['From']
    if pd.isna(source_raw): continue
    try:
        source_id = int(float(source_raw))
    except: continue

    date = row['Date']
    subj = row['Subject']

    # Clssify recipients by typ
    recipients = {
        'to': str(row['To']).split(','),
        'cc': str(row['Cc']).split(','),
        'bcc': str(row['Bcc']).split(',')
    }

    for r_type, r_list in recipients.items():
        for target_raw in r_list:
            target_text = target_raw.strip()
            if not target_text or target_text.lower() == 'nan':
                continue
            
            try:
                target_id = int(float(target_text))
                
                edge_data.append({
                    'Source': source_id,
                    'Target': target_id,
                    'Type': r_type,
                    'Timestamp': date,
                    'Subject': subj
                })
            except:
                continue

#Full edge list
full_edges = pd.DataFrame(edge_data)
full_edges.to_csv("enron_master_fulledges.csv", index=False)

# Create the weighted edge list by counting occurrences of each (Source, Target) pair
weighted_edges = full_edges.groupby(['Source', 'Target']).size().reset_index(name='Weight')
weighted_edges.to_csv("enron_weighted_edges_1.csv", index=False)

print(f" edges: {len(full_edges)}")
print(f"Unique pairs (Weighted): {len(weighted_edges)}")

Communications after cleaning: 165039
 edges: 983066
Unique pairs (Weighted): 228069


# NetworkX

In [7]:
df = pd.read_csv('enron_weighted_edges_1.csv')

x = df['Source'].to_list()
y = df['Target'].to_list()
weight = df['Weight'].to_list()

#edge list
edges=[]

for i in range(len(x)):
    edges.append((x[i],y[i],weight[i]))

In [ ]:
G = nx.DiGraph()

nodes = np.arange(1,56955,1)

G.add_nodes_from(nodes)
G.add_weighted_edges_from(edges)


Network Summary

In [11]:
print('No. of nodes in the graph: ', G.order())
print('No. of edges in the graph: ',G.number_of_edges())
print('Network density: ',nx.density(G))
print('Is directed? ',G.is_directed())

No. of nodes in the graph:  56954
No. of edges in the graph:  228069
Network density:  7.031134762715852e-05
Is directed?  True


Degree Summary

In [32]:
# --- Degree ---
print(f"Average Degree = {( np.array([x[1] for x in G.degree(weight='weight')]).mean())}")
print(f"Average In-degree = {( np.array([x[1] for x in G.in_degree(weight='weight')]).mean())}")
print(f"Average Out-degree = {( np.array([x[1] for x in G.out_degree(weight='weight')]).mean())}\n")

# --- In-Degree ---
in_degrees = dict(G.in_degree())



max_in_node = max(in_degrees, key=in_degrees.get)
min_in_node = min(in_degrees, key=in_degrees.get)

# --- Out-Degree ---
out_degrees = dict(G.out_degree())

max_out_node = max(out_degrees, key=out_degrees.get)
min_out_node = min(out_degrees, key=out_degrees.get)

# --- Print Results ---
print(f"Maximum In-Degree Node: {max_in_node} with in-degree: {in_degrees[max_in_node]}")
print(f"Minimum In-Degree Node: {min_in_node} with in-degree: {in_degrees[min_in_node]}\n")

print(f"Maximum Out-Degree Node:{max_out_node} with out-degree: {out_degrees[max_out_node]}")
print(f"Minimum Out-Degree Node: {min_out_node} with out-degree:  {out_degrees[min_out_node]}")



Average Degree = 34.5214032377006
Average In-degree = 17.2607016188503
Average Out-degree = 17.2607016188503

Maximum In-Degree Node: 3266 with in-degree: 764
Minimum In-Degree Node: 70 with in-degree: 0

Maximum Out-Degree Node:1219 with out-degree: 1284
Minimum Out-Degree Node: 9 with out-degree:  0


Subgraph degree>=9

In [31]:
node_list = np.array([])

for x in G.degree():
    if x[1]>=9:
        node_list=np.append(node_list,int(x[0]))

G_sub = G.subgraph(node_list)

print('Sub graph of degree>=9:')
print(f'Number of nodes {G_sub.number_of_nodes()}')
print(f'Number of edges {G_sub.number_of_edges()}')
print(f"Average In-degree = {( np.array([x[1] for x in G_sub.in_degree(weight='weight')]).mean())}")
print(f"Average Out-degree = {( np.array([x[1] for x in G_sub.out_degree(weight='weight')]).mean())}\n")

Sub graph of degree>=9:
Number of nodes 6794
Number of edges 144885
Average In-degree = 114.89137474241979
Average Out-degree = 114.89137474241979

